In [ ]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
import time
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_EMBEDDINGS_API_KEY = os.environ.get("OPENROUTER_EMBEDDINGS_API_KEY")
if not OPENROUTER_EMBEDDINGS_API_KEY:
    raise ValueError("OPENROUTER_EMBEDDINGS_API_KEY در فایل .env تنظیم نشده است.")

LAW_ROOT_DIR = r"F:\Thesis\project\2-RAG\raw_laws"
OUTPUT_DIR = r"F:\Thesis\project\2-RAG\vector_store_bge"
MODEL_NAME = "baai/bge-m3"
BATCH_SIZE = 100

def find_tsv_files(root_dir):
    tsv_files = []
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith(".tsv"):
                tsv_files.append(os.path.join(root, file))
    return tsv_files

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_EMBEDDINGS_API_KEY,
)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

files = find_tsv_files(LAW_ROOT_DIR)
print(f"Found {len(files)} files to process.")

for i, file_path in enumerate(files):
    file_name = os.path.basename(file_path)
    print(f"\n[{i+1}/{len(files)}] Processing: {file_name}")

    base_name = os.path.splitext(file_name)[0]
    output_path = os.path.join(OUTPUT_DIR, f"{base_name}_embeddings.npy")

    if os.path.exists(output_path):
        print("  -> Already exists. Skipping.")
        continue

    try:
        df = pd.read_csv(file_path, sep="\t")
        texts = df["text"].astype(str).tolist()
        print(f"  -> Loaded {len(texts)} records.")

        if not texts:
            print("  -> Warning: Empty file.")
            continue

    except Exception as e:
        print(f"  -> Error reading file: {e}")
        continue

    all_embeddings = []
    print(f"  -> Generating embeddings in batches of {BATCH_SIZE}...")
    try:
        for j in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[j : j + BATCH_SIZE]

            response = client.embeddings.create(
                model=MODEL_NAME,
                input=batch_texts
            )

            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)

            print(f"    -> Batch {j//BATCH_SIZE + 1} done. ({len(all_embeddings)}/{len(texts)})")
            time.sleep(0.5)

        embeddings_np = np.array(all_embeddings)
        if len(embeddings_np) != len(texts):
            print(f"  -> Warning: embeddings count {len(embeddings_np)} != texts {len(texts)}")

        np.save(output_path, embeddings_np)
        print(f"  -> Saved to {os.path.basename(output_path)}")

    except Exception as e:
        print(f"  -> Error during API call: {e}")

print("\nAll done!")

In [ ]:
# ===========================
# پردازش سه فایل آرای خاص (که ستون vote_text دارند)
# ===========================

special_files = [
    r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\dadnameh_metadata.tsv",
    r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\vahdat_divan-ali_metadata.tsv",
    r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\vahdat_edalat-edari_metadata.tsv"
]

print(f"\nProcessing {len(special_files)} special vote files...")

for i, file_path in enumerate(special_files):
    file_name = os.path.basename(file_path)
    print(f"\n[Special {i+1}/{len(special_files)}] Processing: {file_name}")
    
    base_name = os.path.splitext(file_name)[0]
    output_path = os.path.join(OUTPUT_DIR, f"{base_name}_embeddings.npy")
    
    if os.path.exists(output_path):
        print("  -> Already exists. Skipping.")
        continue
    
    try:
        df = pd.read_csv(file_path, sep="\t")
        
        # استفاده از ستون vote_text به جای text
        if "vote_text" not in df.columns:
            print(f"  -> Error: 'vote_text' column not found.")
            continue
            
        texts = df["vote_text"].astype(str).tolist()
        print(f"  -> Loaded {len(texts)} records.")
        
        if not texts:
            print("  -> Warning: Empty file.")
            continue
            
    except Exception as e:
        print(f"  -> Error reading file: {e}")
        continue
    
    all_embeddings = []
    print(f"  -> Generating embeddings in batches of {BATCH_SIZE}...")
    try:
        for j in range(0, len(texts), BATCH_SIZE):
            batch_texts = texts[j : j + BATCH_SIZE]
            
            response = client.embeddings.create(
                model=MODEL_NAME,
                input=batch_texts
            )
            
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
            
            print(f"    -> Batch {j//BATCH_SIZE + 1} done. ({len(all_embeddings)}/{len(texts)})")
            time.sleep(0.5)
            
        embeddings_np = np.array(all_embeddings)
        
        if len(embeddings_np) != len(texts):
            print(f"  -> Warning: embeddings count {len(embeddings_np)} != texts {len(texts)}")
        
        np.save(output_path, embeddings_np)
        print(f"  -> Saved to {os.path.basename(output_path)}")
        
    except Exception as e:
        print(f"  -> Error during API call: {e}")

print("\nSpecial files processing complete!")